# CC - indirect · Kaggle GPU Worker

> Run this notebook on Kaggle with GPU (T4 x2) enabled
> This worker handles heavy AI tasks and connects to Render via ngrok

**Steps:**
1. Enable GPU: Runtime → Change runtime type → GPU T4 x2
2. Enable Internet: Settings → Internet on
3. Set your ngrok token in the cell below
4. Run all cells
5. Copy the ngrok URL and paste in Render env vars

In [ ]:
# ===== CONFIGURATION =====
NGROK_AUTH_TOKEN = "3IRWywio1R6ggk1emVwuJOJdmw0_5Ng4ZMPK1peMjXbAVrTci"  # Get from ngrok.com
RENDER_API_URL = "YOUR_RENDER_API_URL"     # e.g., https://cc-indirect.onrender.com

# Models to load (T4 x2 GPU can handle these)
MODELS = [
    "deepseek-coder:6.7b",
    "codellama:7b",
    "qwen2.5-coder:7b"
]

In [ ]:
# ===== INSTALL DEPENDENCIES =====
!pip install -q fastapi uvicorn pyngrok ollama python-multipart nest-asyncio
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# ===== START OLLAMA SERVER =====
import subprocess
import time
import os

# Start ollama in background
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)
print("✅ Ollama server started")

In [ ]:
# ===== PULL MODELS =====
import ollama

for model in MODELS:
    print(f"📥 Pulling {model}...")
    ollama.pull(model)
    print(f"✅ {model} ready")

print("
🎉 All models loaded!")

In [ ]:
# ===== FASTAPI WORKER =====
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
import ollama
import uvicorn
import nest_asyncio
import threading

nest_asyncio.apply()

app = FastAPI(title="CC - indirect Worker")

class ChatRequest(BaseModel):
    model: str = "deepseek-coder:6.7b"
    messages: List[dict]
    stream: bool = False

class CodeRunRequest(BaseModel):
    language: str
    code: str

@app.get("/health")
def health():
    return {
        "status": "ok",
        "models": MODELS,
        "gpu": "T4 x2",
        "ram": "32GB"
    }

@app.post("/chat")
def chat(req: ChatRequest):
    try:
        response = ollama.chat(
            model=req.model,
            messages=req.messages,
            stream=req.stream
        )
        return {
            "content": response['message']['content'],
            "model": req.model,
            "provider": "CC indirect system"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/run")
def run_code(req: CodeRunRequest):
    import tempfile
    import subprocess
    """Run code safely in sandbox"""
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix=f'.{req.language}', delete=False) as f:
            f.write(req.code)
            f.flush()
            if req.language == 'py':
                result = subprocess.run(['python', f.name], capture_output=True, text=True, timeout=30)
            elif req.language == 'js':
                result = subprocess.run(['node', f.name], capture_output=True, text=True, timeout=30)
            else:
                return {"error": "Unsupported language"}
            return {
                "stdout": result.stdout,
                "stderr": result.stderr,
                "returncode": result.returncode
            }
    except Exception as e:
        return {"error": str(e)}

print("✅ FastAPI app defined")

In [ ]:
# ===== START NGROK TUNNEL =====
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start tunnel
public_url = ngrok.connect(8000, "http")
print(f"
🌐 Public URL: {public_url}")
print(f"
📋 Copy this URL and set as KAGGLE_WORKER_URL in Render env vars
")

In [ ]:
# ===== START SERVER =====
import asyncio

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

thread = threading.Thread(target=run_server)
thread.start()

print("🚀 Worker running on port 8000")
print("
⚠️  Keep this notebook running. Session expires in 12 hours.")
print("
💡 Tip: Use Kaggle scheduler to auto-restart daily.")

---

## 🔗 Connect to Render

1. Copy the ngrok URL from above
2. Go to your Render dashboard → Environment Variables
3. Add: `KAGGLE_WORKER_URL = https://xxxx.ngrok-free.app`
4. Restart your Render service

## 🔄 Auto-Restart (Optional)

Set up Kaggle scheduler to run this notebook every 12 hours to keep it alive.